# ZeroSight ML Pipeline - End-to-End Runbook

This standalone notebook embeds the parsing, validation, windowing, feature
extraction, and training modules it needs. It handles dataset validation,
isolated candidate training, artifact reload, held-out metrics, complete-window
inference, and a saved verification report without cloning the repository.

The reproducible training entry point is `detection/train.py`. Each input file
is a separate capture. HCRL `R` rows normalize to `Normal`; `T` rows normalize
to that attack capture's class; explicit supported class labels are preserved.
Each capture receives its own chronological 80/20 window split, and the
reference CAN-ID vocabulary comes from training frames only. The streaming
validator rejects any capture exceeding the default 0.1% malformed-row rate.


In [ ]:
# 1. Extract the embedded project and locate the dataset
import os
import sys
import json
import time
import subprocess
import base64
import io
import zipfile
from datetime import datetime, timezone
from pathlib import Path

BUNDLE_B64 = "UEsDBBQAAAAIACMSUl1UUnmaZBkAAAlgAAAhAAAAYmFja2VuZC9zZXJ2aWNlcy9tb2RlbF9zZXJ2aWNlLnB55Txrb9w4kt8D5D9wFfhOvSMrdnZmsei5XpwncXaDcx6wvbNz1zAEtZqyNVFLPZLaTs8gi/sR9wvvl1xV8SGSouTOzh5wwDWCuFsii2S9WC8yCIK3F2xTr3nJWt7cFxmfs7JO1y2rG9Y1aVG1LGWrtOVlUXG25h3POniVVmu2ber7Ys1bVlQ5b3iV8fjpk6dPPpwcn3wzZ39pOXZt79KGr9lDUa3rh+OS38NIOU+7XcMZ/wQjELhQAC7q6rl82cbb/ezpkxxeruruTsylqG5pZD1gxG53aZNWHef4TgHepk3R7dmKdw+cV0+fWJ1pndVtzN7Ssje8S9dpl7KGZ3UDC8cfLe8iDe2eNy3MLHr6pN2WRcdaANfx231E4ATyZBuG8204YGa9y4pVAc33gJQgCBAzxWZbNx2rW/31x7au9I+yvr2FienfabuvsqLWv+/S9q4sVoCUpt7AGjv8xeTLD/BTvoEF8K7YcPVK/Y4Y/v9zXXHZsNtvESmy2fst4j8tI3a925bQ+lWRARbOKljoRdF2xgp+rFc0Efmz2m22e5guq7b62RZwA0/g33Yth2s/ljxtqphXLd+sSj2/S2hab17XQPTuZZm2bZEXvInYm7YuU5yTeOVAAbo1RdYqIOHTJww+aZbtgKf2SQvEhDVsgaoFUkY9gN9pWapf+an8JnpndZXvqPUmBeifIpbJ+WQ0kQRIC4M9fTJDZAhkK8aNFeM6M/orcf5r8fJccbwc8PX52fVfLs+T788vr968f4foch4lV9eXEQ1Iw63S7COv1rEUVpAS4jb41hqkBP5veZKlVYJSG7GqbjZpWfzMk7usKZMyXfFSoTNrim3XxvfwHhklkeyvQLnPceHP2CXJCqsly7Af/vRdXbfAsvdpAdCJ7yO22nWsu0OtsS3rPWgBrUcaviHVcvk6RunczwU65JifblcCWotfGXsGC/gpnbPXX5+cioYw3vv3V9fJ2fdnby7Ovrs4Zwt23eyAsfmnjG879oYgnTdN3czHu7xOy5bjilD0eANPpAzGt7y7oGdhcN/WWUxSnkisB0T+IodpdUO4cjwBMgZVVYeBwk/Rsl0lkVTybxU5e8TsUGv6BSIWw768OLu6SmCk84srmPAyeEfEDSIWvKqv8M/r3c8/7/HLn0BQ8O/lh7fBjSCco/NQXaEa51UK2pSFH06PT34XMfzzNYz29v2r84vk7fn12auz67Pk387/nYYUCwwkRyRS9+FIUgbMR6QzE6UzA8n4gUCo0U7p6KRNN6B+WnoGCOh/y55iK0k2re7F10na4Xu5SFrFnJ0rPYNiXdyCXkBuffrkL1fnyfm7q/O3ggnqFqnNq/swMN/QcpA/gllc1g/ACTPYd1gYnIpxdxz/7mFigKg3V+8vzq5RXF+/vzwHbnj5/t312ds37+ghDJLDptqFxlDTPRD0SXwazIjkT5+QGhLEu5LbtMDGmucsSQBxXZKEwEN5JDakZF00c9ypYOwgfk7PEGGKZvZrfPq8SR+CmQSLH4QWa2DQELeYUD+YOS0NyKqt8chtTWDmes9Z+ln+BiC9o/3K6lyojSHJqYMBx9kzRgBYzKfxAAiPT44FewWDIRPJanOwR+qyVx42YLEpGRPCTXQJA9BGejMyn6Sss4/wSu748QX8DF2Uab6fC26C9qcnJ/GJ00yJoDaujMn496KRSSlAVbrh5orQFsAVjS1GIFepmDmzMQB9fvns9ACllyijQKPXlEUUAcXtQAk0UPma2H3Gjv9IPQzGbTjMuzImg2oXVbWYrDQUENUCIAhPQbujgDgqAvHmI/wfwubKq65d4H4TgQ0L6Ejqj/TTIJnohUYarMURpecsUAo/afJ4+7E02K3Ix3u5jO90ldw33l82iNHstPsRrSY7UgvV8zGeAyB+Xgs1Fy9spp6ZQGFr7fEXE4rb0CQMfnqzwfwIgi+kiRojq4Q9rNmwA2wXwmH5Xlo6yu4/brM7MFMcZ4K16T1YMmnTFTmsiFwAvewhdFyJid2xxajPQwEUqLe8Cq1uM7SF8pE+DqfqzX1B7oVAQe5Zue6niaA3Kg843LlCa/N1CDg1gqVwbQ6z4btmwbD31DD/GD70gkeqJ5ZKnFqG1RCWsbwZgdtw6cmStV6sJ6EOGgcjUIHrhoBBCaICnGCiZ+w7sEcfUrDss3oDfCc9WDITFce3LGtgdWjNc9RCU+D0JNibVy0DG4rDXltjPIB/ArMwK7oS3eMxAD7kLMeb4wcRvixBfIItb6BLktW7qkvAsLmZ7ohLxM5o43mIPd0ZXQFoFbcdYKlFGR6MPw5gZGJ+ho79DNOFg+eHyIni4xJ0UuhZ9QyjQPYWb36ese95U+T7XhtiiEJwH/qHfD0lTNR2gtsVUGo4wekGNB2POUjX4gfg79JSzUVGWOL2Ln3xze+NfQPQnq6T1R4cknA2i+/4p3VxCxtwOEFXmJoJ/TcLY6YTM8IPWJotZ5fAOcWGkxMbTnfATyB8O5sWm6LdpF12N0eJ4xmK7VEbsVuwho7amAUHgD1TADcp7oKgGppmt0VIGCIE5wz4bh0fAOmIhT0GIhM5E1jEz8jrbNegJebXykPJsZqNEU7zkw2U4oae50BU7ywm6PvFtA1e2/ZIT1LHFBEUbtnRmulY1CNkCcIjEHKM0/RbpghechsOtnuUXYJLTk6SsGIw8tOiRlVxVMsNf4xhjtgBLI+K3qe2ose7erot59/cHNATB/US/ZBhvR0PGHhCQjyvkIuHen7aAPBuDH+vKCkDPq7UStukqBKUF6LZYKwxHf1F0pJLFagE4RffHD73TI18r61lcAXBH4NOIxP8PMKwntX7tjTYo2GEdFdqIzctb2vwLO42gYzZxElCcZ4kgW84aJIcBnw5gIhu9hjMcXYx/XCRZ8kP2kj9npg1YddzdTw0OdAEl1vh1AsKAPQJAiaiPYwi2kdocE8DlEHic/qD/cG7gmcTi5DDg22MMcoweJ0WJcygqylbNpjJXMwCYI4Ih+Gpf5FLqHsd6BHKJMljruAzdlaWKtaPy9gCz8D6/vs//4thMO62InUunGuystLmI85ABsWm2FSx4kibPrKmY/gjyLdoT0jDlIcYo6f8pLMvQkP2c8kMpo7xoBVzHsdCG4l9+LllXzGO/VpC2Lqu/rlj7UNalvUD4m3jmQhC8s7jAKZ8hBllbNXgwB7EM9gCBJ0V6RRRVVKClpDDD9gic17uMXvE1rvNBtxAWBFmLIxQ22PqOYDh7IFycIjAaBH25zWZC8q6YqmajDJp8qIBsXY0bwDqmbP/4E19Vdzedcnbi+RDsaVwWlxs99UKLJVr4FaVXHkogLcx/ofpG44pwXVBKbuBeRSgrNWwK6Q+HLkmy5HBa/2bmRtfFOydqIjfIMgYBIFEg5X46XNCxNsv04b9GRaELP/y6vs2ppyygvH/JdpoaKjFIDUZunmImblOo+8y6OoOXI+iomfBjSNjjnwNBTB4JZOkUk2ibjyKvwa9npaAqw1fHzcg+mWxKTqhv4/W/TsG79pvsRPIDn4XHLrCgois3IF2CzzWoDn9TfpJTT4BAAnm1oKbaGqJDkQbNSgcZl8/Wg6yxTRqDFLlpJ++xXjOjtLUnVQKc59H8ZXYq0DlbNuwhyKCAj+By1h0+wRBog0XmIuWIcSCcuFJXpQq9ubGJi2VmLX3ou0wuoQZVcVdMbTT2VX34S1PG/uZA+jyw1u3j0jKJ82uojdx96kzsGFEhShdn2zSLeZOHpngXGaBnWbDSc91nthpOljKXCWSH1nSXCSa3Wa+Zc6Zzlr3rT+bRMkbkXWiZNN2HSNXvcZnaNkuDeRg3C5XgTtNSodzc1PVmInK56Kv3VoKRP6YceZuw4rzcQYCAm51ciPORywScIrBjNlxc/Ga6BiaU8Rf0kydYOE6V6uipBjF/gAJIY0Wic4z7+rWecw3285nuK/zZZClW/J9UA1gVmCIJUGhON2CZboO165V6bPZjoCgR0L9ybktBG4QekSeIcDppz3QUZIr/h61BHbI2qBPK+0QYBq0Q16mFcLv5D4srBBlDHh0VPDnl5cXuCcfqz1ZSQLYbyLORnQf9sQgmMOFoAvgX/xjXVSh5uBJlSXTOyAXwD5Z2oUCMeD23IK4AdWqNf/kZiQtilzj7qAmrakSSCLA45ljNl5R6RtPwdSVvEECd9fUVQ2QiwzAYci3kWkBMH2a4p6UPWyOL8/esTevTHj3dZaudmXa7KlI655XZM1TrZ8aAO1ClYzr28MU04/pLTcSByJ7owMWyWpXlOtE5nRkXZ9YVd9HWGWyZ8So1MSFI2pXlCykIOlcJo5aFxLuOCIa/4tHTpZB2qyKToS+SKicRqDFBGDEqjW1YUOCyCjMhi2WgaC+CdPNsH9BYswebyxNFg2TMwsTDwN7mCb/K6NLPwhTOmJ78UXTScd30FRWiAsdCtvhHAsq0J6AisDEJEyDTcZBgrpCOZLznbHFgp1gtFw+BBji2eOaLHjHOXlLwPQwue6h1swORpTIxIFqrI5JmSsFBnJDayH+bQNLlE014GzV18ImO0JvjGquInYtghn6CQtxbsfoUKwF+Jm745srj8wlRxZTOM4L4N1fA+RMskrgbQE+N3jbi9OTE2dwtIzX4D/fLU5/774CZpCrEIK9+MZpISJlDxy9ygU4TSUWxQ2M8YammbQdoH7x9YvInd+P9apdHJ9GXhEQjmRedKHLzBZK9gl6qTqIJ11WhciBD6aZFtO2O/CI5PNwr3kbQag4I60TsDAEtByW4eH2X1RdiIRUU/V2NGv1nE446dmgamQkOvRYVMgJbMlvIwpPqRonmWvtbBh5oDlECpis7JZuaN9YhizRNQlVD1+46Rl7KQjhZONQewNa+C1VlNxj8lRWFruu/D8iIekPHIKp8RB4o4fa7Qo1GnIwKcCWqLrFixH6meUmrnPiFoiC2U/1xxl4GHfCbkruTwceg1tEOh+pibY6OWWmMJS1bSeWpZL84SR5cTIY2C4+mXuKT9weA2mZa7ZXKtDNCtmCYrUnMXGb9xU3c6fixjsZUQs710X/cVU/hKruP9512Qxj8RiIALNxMNi2TDN+V5dgvCHascJxjDqitGVuS5bbeFixAouom05W7x1S3zCYo10XMDcEZrKlkR0B5hACNeAAWXme6GgotB0UePvZxgLv38ocb1d9Han5OkRQPXI4LrQTO/93Ktaoi1zVQYoFxrXooMQmzZqafrpo03uA6oSxKP1QdbViUI53ISu2RRx0kNaBbyKKoG0fKkiqani0709gWPbXsLDU2adFWGKTtlh4qw3KhY5MeLxm3SNud5twxv44sODw80MiJ7tQBuvS6Oqp9hlLkDm1zCOJTwwgpGDbUMvFdFn5SJJ52qLRGPNYNgY5D1qVtHwEPjydzA3WC2AqszfO3+oTDFhLGRnwBIxcm8t8UVj8oMYeMIMv+X9o9laxKJlN/hSYt5+LnMTiBYI2zQ/Dcb5YF2nKfblOouSIFbYSBxgoeDWnIwQyFkQl+lTqbYYCzZJv4bWbgcF+JTIYFjT+FaAuId0H8u9bHb1b0J8YplFsRwoeMICEjUYQ1Af63Dfj6XM6x4UWsH2gK8T/RlLb+qjXOlEhRN/pr1CAXgb0C/W1L1aoPoheFen7xd8EPwHaGS2WhMEeqAbon03VuLjxkb6/GziZArIuM6Mn/ppujuxNNrTZq3842Vkgbj5A+Eifzx7Uyoxv+H1a7oS/H7E3GLqj72P1AENGkgceTNkIGxGjMiTNGxQTErfO51ZvEjUKvTvHV+y05XcI0YkBqh364Q6j4DAzefL2jqqKZWBvhbHXtCm4k8YEKRqJS8slmgIOjc0wNckfiDAAyOpyt6ncODHFyuntdu+KsCfkrX4b01MQ0IJN7pFobWj1i5jF7gw8nPUigJ+o3C3PTMUZ3VRGDywy46ww6G1T77arfWiPh1MRgV73dATOQ5T0yw6WKMZdndCx3XDd7bd8QWX/M/ZbPFJknSnCD9i1txRtE4cDFOjliVvVvtplH3mHY1bbGBrXTahbs2MJZ8aeu0cH4rTFWYTQCzbX3389G2poARpxAY12VfHTjodyOJ+gSNhEMbn+uCjrbEkTS7uqrn7mTR3qGS/kCDOPgUZyjavSULXeBDxS8bRHuiWRH9GbJjnnJvlHmotZSr9RTnmsrZwuVaULB1JS8iuzNxB90q/U4GSYed6jAbmIAlOB1CfB6FyUutykn0LM1QuczkBG+H4hfsRUK++DYCpPqQl0dFy8+Vc0XosMDI67em0oPV8EfyCHc6+ui2QcPkfPlM4JqmN3J/EfBAzSk3RafumH4HvqKNGXpvoEd0bcMcDvebNX7IDB9p92RUPZFDvCSzuzpUOn8xrLm8jSOXKE1jmWZ+cjFJqcaKnsapY2qoyEqUspA6+kQDSYjSk9On0h4gMKvCvdqnG/JtVy2cO5mexDCjwUjLdZrVO5wLlOqLiC4+o5Gd93wM7Yv7AXHl10cA1rHryUKv+Xfim/aT5jGJDl/IE3rLtLKzML8C2sjVKXMhWQquzYMVZvCWbyZCGd9QgpId5G+TyNMFgeepd4zE4jHbMavP2tIzODyguLO2P+qUOmcMAs52o+Lt5NZh7rrPrOb4ZKY0I2LFPJn/uxw5/RY9rDOT+rj80OVAdsSWgNNSnW2unvpppQt1HojcgV40PkVZc9jybe5De5YA+TugnHaFxi7M4OHcV6lE5YCrNCX/yCVTyiIgAskdnMOpplIdVlD7mNWKpG79RDbgBcpy3hOlQIFsaQMF7S7ncvZpHZSu5ZFqvwCjZUzhvNM1Mm9RS9o55DnB3iXA7hvUSnFeWBKdMjsTqnfLss0DADbebVPP70LyFb3tnTn9vAXIaosOph0VgtS/FMH2lsttozbWFGjOxVjhq909pK5FWoiEA86cHpUyd1xY2bd2hOhHc5AFYhpz/WlEARtyIUojZZAIwtzA0jgsNEuP9UxP/KsebDyhRyi1NzVVUwUkfgF/3pOoFHqwO8pwzlXCyc4s7j1zYiH02OABrabYi7yuBoxWjSf7BxSJklsEIoSST19Sc3s+iL6wl+dT1CxPaP1As8Wiog14WgIn+WktSMm9gVSgbD1ngJgUrvysQu6Rp7PzKok2YZXjVhXZQU2pBmplxybO7coxS6I6dgr4LVsQhEqD9iSHTwDe6p0+LEWnFG3NTfw/RroOWn6Jee/npA2QZNSecCqAFAsQUszEt4TGTBPiCU0tCcLpA8pK04ON4c85ShBca1c8uWxgZQchbopGale7AX28ligVCVDfjadVuZkwfZwYi1hv9PGoQrc7nT5W+H9KlGh/nbWB/NXNAVpvmchfD/VzD8DHVM/+OP7IQB9jkbRCgENzndK7N7NdUd+Ccju+oFWLD9bH6r4ALI/ulX8qmA7ns+PpBmkCWMdzNMmeMn0CDBXW4wUtYPErGvff5xIEbW7cXPscb5qW4oFj7WsN1t8RYtGWxwCDrIeHweKDW3HkCnB/XwZPjB45lnCj0eZP7Q7oVvvd2kZvH1GeuiU5R28/zUPy9FRIwaq+9uI1eTYHBno6NFg9ZCcUMjUyUcUG3g1j+NVRfshzT7bG4wyvR3rFirDNvxcGiX6e1Xx2YV8KZNVsTFgdYqHSYWZumI4drW9h2R8prHgYXJCkxDUaxamZm03T6n6tXnIqwrNLZuqFxOc3Znze3AwzKwhesSBhgGyFqRjvrI9/DaMJJta84jg+syE5d2iZRETOf0xDiAj3pXrulUFoGHrQWNZ20I99AuSR7d6b62rWy8rZG9eKVuc0SasvAUI0wRe6dpN/s/bWFP1Yn/I842956jbdTa1t3M40gODzfgFESoS1VvGxEpejQXf8Sxlj54L/1iu4ICpISEhKdNWYizsMh8uheSV7oJ5KQbAUPVhAILuX804VCQB06TtRKtluuPMaSiCnuofVYByWOM1m+Rtkl+eIRCIc6ZwqEmvniIuuMLWcG8YWzYrasTIVLEFh7bXwDqTwjK8s5h3OVxJTwIJtErYrqITSbvPohBsfJBBUAndLHUIGTvyNOTkbjQYS3u4dXuD74dVRJizaQL+4vvHDmVMuyWgSoy6RvdRu4y8BVRi2sB9AkPAU3cIudeLPODYpLBjiiwMjRdJTqsk4x9xe5seXLjdqlX4orUwrlESvVKqImvb49xnY0DHYMxWwuqrRzwM7wU8xjvGvUcoTcqq9CnsrZQSR7/ZQH+Mibj6j0PseRIwntTuBgUDa2VHZjvKsK1DzU0tzZRcx+DZREGPaXjUw+c3AQl7rru6dzXiI2VbZhMEZwJMCPXR3goiuQ0RStdtaGFqNmQvofpuQODLxN7OenswW1mlvD7vZpK7Ge0wB8A9cviZqb8YhXc7R3jsWgJfj57t2RQc4Zumk0op1H9a9q81NK9u9KrZqfUKp5DLXmvU1m4ci9YK/nMUrR/bdKtNHdFZx3F01oa+RG0Bb8FTNExEHv/GNO9MkQEaIlYghGgh7SQatbZgJY9Cjxh8x7O+E62wjsRDvUh6MUArSZegXS7svPVS2jseE2+fqre9QqVPjDxaCyVRQh7GLMhLmRj05MC88mK0tESx66DHbu1VJ1p0Jc270ounSjBFh3wV1GBwUPXNvdXmFIGFe/WjsGGL4uct9sUj5ZY91cDMsyrhNGy+R9QSwMEFAAAAAgAAxJSXfjxlkf1BQAAlg8AACEAAABiYWNrZW5kL3NlcnZpY2VzL3ByZXByb2Nlc3NpbmcucHmNV21v2zYQ/h4g/+GmDZic2KrTDEOXJS08J1mLuUERu/2SeAYtUTFRmdJIKo6b+b/vjpRkOkntJYAokffG514dBMEnxQuVx1xrIe+gNCITRnANaa6g37uCaYnvis25jvb39veGM6Z4AlNuFpxLSLjhsRG5BM3VvYg5MJl4u/3BB9CxEoUh9iAISIaYF7kyIMt5sQSmQRbNnuL7e6nK52CWBdlTbZ+L2LShJ5dtGAhtSMj7/vVgMuj9cTEYwhk8BtdBG4IRPXr0+JMel/Q4p8dVruYss5/50B6V374tLSFnitbrTx+D1f5ebzTq9f+a9Ae94fDCSd7BQf8JT0FaFeIbn8xilU0yNuVZqNjCvZ2ANgr+hatc8jbErDCl4t5RCzpvaT3Z3wP8Q6SuanlAVwVLqmExExmHQnGLNyKk8kUn4/c8qwx4xYxh8VeYM/WVK+c0kkhCftZwXR3AnDOpgVVcJCaCUb0roZKC29ajQiNfUaDnTe7kmRkHiWGR1KToMQ55igfM1BeM4OKhyEQscCdjWte3wBiq78CTyAkcocBSWgIUWpll+EMjjIwgrbnMlqDzUlG0ZVm+sFZBPkctzGmIGhTdi0ih8QScnaHvggpo+lMcxcsmSL7DMvJZ8HTDh2ivASFhM3w8BquHCc3hC8tKfqFUrsLAenbDYegDxf8pBYLjuaFGwIIYtJ6ZvmHLC/ajZY//NweeA9PIeUmytiFtg2QTEcKsUrkL62fApMFH4QoSVqFS6rKgMoB+toBRUFaZ89hY8oNaETB1NhZMaT6JmZxkQvKQHus8o3Jygx+2pIzXKfeJmDCGSDNmGTGBrUZ9puA9OoIs6g+/UHGcM9Pk1qX9PAEjsEwaNi/aGOFTYRSjIjgRSRuSLMYHM2wyXRo+OYqiqzZ4oDahapWe2SVCE0URthrYKcjsTTw8n8XUxbwwS0vm8CCqH8GPtMpPLg3xOr+7ZHYZV4daqYkA6nykRKxlpdV1R/V1TyCKIvhw7tZutwvng777OJ3xB6Ar67dVUpKkCfLHM7wmlgj7Gno3Cv5eC77VB+FNt/NbxC8OO+PD1q0+RDX1Luukvc6l274dHuKT9NLpbUKbB2F08I7Wn4L2WgFBU32uoV1b5YG7y58FW2Y5S/Aea/boTuVloUMvSTEWjUaim13yDiqBkcaiacLW2MnAium7vBFHL3WM2H5NG5TrLnisjKAd1GLwlhmXoWVvwSkcbwujNBjlOaR8AangWaIx1dbMq3VkNVdCg1K03TiSm+54M2xlEQmdCikMDxueFqX3WsIpdLdGdhMWMC+1wRkEnEBbfGQuO5LfIaL3vK6QmyCjhc62o3GNWkTdQ9W+Mmrp6afaYZmENOFTdx39WvHwh5gXxjOTBhrc3I7tB3mPrT2xAxalzeOmfMTX1R0U1KBY2YMgEWrV11voPhxdur/tKp0qyEtDTVoxecdf0utkYDRWF3eIvX7izS6cnlkiXN5s14sZ2birnhq71mNvUD/KWPll6jOyEwed28CDOC+lof6e4OCH9bcUekadMS/IaCxYrttYuBgFBMtqYXV6UvnBwSnXNDvwTowjLIsN9lqqTfdkLMxouMHJB/WgnTy6i2iOrCpWU7YxC2j8vEFgxpSAVWJV/c42wpdSDTvhMRwSXh5Utg3UqexQPj5pEv6ZgF+eCai1Ot7O0XhjOtmYSrxZeddIkgaf5VeZL2TTYe268oeOF01vDHhSrb4TEjg3WOe38QeGsR19honj1RjowPGqduGrysm2FK3jhWrelC64tshHGK2bbvZQL4Sx86Rllrnuo4Lw3Un3ofXO6yqPR+3XK5yKpq3dmPn57EcdwjetBpOaex1MkY25JKQ0m9qS0lysmpIe12xBUyYDf87wCDZTGame1CyPFIHHc2o5/mZjGJ01Hz6J9QKe2rU68H4B2Z+JE5NPEpyvQvu1ddzq5/KeKxrai2XCpBExAXhJfJSIuJ0xdC5Js65OObOjCfZaSmC8V9SMTTsQs8ZE/w83R7sLPUe1DcOKYiuSjuYJnv8BUEsDBBQAAAAIAIeeUV35taZJdAkAAM8hAAAVAAAAZGV0ZWN0aW9uL2ZlYXR1cmVzLnB5xVlbb+O4FX4PkP/A+knqKh57FtMHoxkgnThogEyyzaVT1DAE2qIdYnWrRGXiDrJ/Z/9Hf1nPISmJpKjMBQXqh5mIPHcefjw8nEwmF4yKpmKE5XueM1bxfE+yImlSRnZFRT6cXZNNU5NdRTNWk6BiTU03KQunx0fHR7/MTmbvFuQyK1OWsVzUhJL6kVYsicgTq2pe5Cwhn3meFJ9PUvbEUilw1+p8FhXdClDT1Cw5PtocCAzwHGyICHuiaUMFiIgIzRNS5CkYSHi+YxXLt2xKtOmgFGTR/b5ieypAXwGqj4+2Rb7j+6ZCa4ngGdN2gBMJ29EmFeS3+WxGsjokJauIeGRkkzashBAI8p/f303nJ/jvz+DpZDJBd3lWFpUgeZOVB0JrkpfHR7uqyIg4lBg3PX/OtyIiZ/khIle8hj9vSnSDphG5YwIFXSzP7h9ul/Hfl7d3lzfX5JRM5tPZdHaiTARl58uLs4er+/jT5fX5zaf44x3QgLXTGbL/5exuGbcyPtxcPXy8xvnV8RGB30QJiWtBKxFn9SSyx1memKOwrDXds3hbNLnoRpuc/6thMU+c8bT4zGoRb2kOc91oSQ9pQRMQLaqiPHTjm4MAwY80B/kVrE03kaTbOGM0twZq0UvEBQMHsjJOWCqoNngtk255G1+ex7/cLi8u/4Ghg9XrDI0hdg/Xd8vlNdJ8uHm4vkeSJq8Zy3t3UM7xEeQB0a7Eoohh3QP1tSC1qEJy8h7STSyUSRWDZMtJTxSR+Z9CU1CRlQ34K53WkQgSCsbjSL2QybAC/rWUvIOItbL5juSFIAaxGjf0yqXHb+lADU7l5XTDc/lpqIlIxvMUdrN4PH37Dg1EprIqNsijmd8QoDCYHCr5/0p9vSeztRUAaXdwAtrrJgsU0R/RmLTYv1XfYTgaFyMZgJY9tUZvm6rStniC01MSXpNrQBUCoIE+9DMh+cOpHDJEvRLGhO926Cs6MZdQRyOygeUl/+blqGlgC0U9m9CKiRL2xlWvgrBNaV2TT3LzachatsCnDQR80UP1GG7uWrCTgAMwy2B/AgySYqcwVQK0RGWUeKHwun4smjQhG0ZqQCaQiRDbbi0YBr8B9RiCVwlWAgHPp+T+0cBmJa+D1+4ogEQuyI4/s+QkAZRFhOsQFvFaLzvYIGCyFnxLUyWrc6VuuJDwjOH/eNWBvx/vu1C1Psrsgl3LRRwH/UrXLN1F/adGvQw2oMwqWPMBuEZmnmh9GhiArYXvFaD3CoBhvQYZmIQGm/Yp1qsnAQSoHKDXDGZeorXTzkZg6f52aAaGYfIyEQzGZZYOqVlaM8kQOoId00GsM2LGW6dFrKwcj7pKEo15eCBi4OShuF4PF6c9qvQStWFCJLB5jcD1e8bYG7iNSQ0plLbHPe4PvTdk6jia1WHYh721hPzkrEwbBAOutY/9uIEIkjlmWSkOcWte4OiIbCNCU0W/xqvddA+rNqHVhgu1z+TRSyazSSg93iFqKWPWrgQ81KQU95zjiWKGP5Bd6zP44UQ21cMn6Jy9prE/T+IUlt1k7mZAxmr9mpAOnUzl3SCaMPUa0UuwChqizgRFZmR+V9+026jdPB4SS1LPaJBaVREQwhEcGPGXO9JcD7kXZ6bRNE31AQdeb6R/iTyN3KDijJrYGFFzCjAQ4i9HOjV2svWHstRvCEZ1PNK2MKh8mTy6HatCZxOAu5ycnpKZM+4qm9KyhOQPcE1tSozQNzIPqfA3XnY4tq84OSHzNfoYDkUZQ64gBEpZC0Htg7VsYBoXqjU3A6sWfWote1sHD2Th7lMy5D7seU1OKJhNRvgc57PQCxO532ghFHlzJ9iiVrC4ElDcwcUDZCCjIH8mczZ/qwSrbBSYGr20tSNoUMh3Jmf0OZB6AOjlllEfRsg9WeCV53hoXgjgTOmPECy+gfzLy4C2t9+kl6eXNHbt8mnsjCBSbdGoISQyguFujDasorbDWttxrW0uy58VaEVjrEEJk9IcAMefyNzL35s1BcTTl2DFBZjcbie1CJ5wtseYGSAVG39oMCyq6jRB0wnIrwzBajf5Yt3qXr4A/8vE60VrxgpY110ujUbD8kQvgT6dtHHD6HTsEJZwgGxy82jc8+wcSZULEEqrij/RVN/V8I7Q8Q1ZXO8gJBiD2BIlsQLvwr3jfvRrYcQ25HV8+z5LAGy+0RCEpe+1YwT+vz9IFmb+sIcOvuAPEsEp3d97j7uhLsjyi9vl3+K//jNWaY4aRsI3vsnfkMDR/0YDyv8wul5TnWA4rRV9pbalz01MMEovPCJhEItpno9cc3pJ1lb2oRHeEBCLbOWDftjCLfYjP4NulC3sMt0ltvtni/ZsM0dDl8ftrrVczviAz+6+tVzW6IDH7c0t3GLRZRg07RaD0sdl6dp5i66i8ZFgg2/RVi4ugafjt/Cc9g6b0+vrA2klpRmUl2EOTZsykQ0pewMYu6btfekZqwPhXPKcK/HYFddOKj36Lfdebcr/N8kBA76e0x4iN4E9JMN89RB5ctRDZaTlyKzKSM+kNxuHdIP0s0lezERpuyo5XkUDlRw836ZNwmKVebDRiiIFBLuvGiazQBag2HIyEkDy420U5gLfU0Do0LaVnWOsQYb3NdsS281hRefHare6G5gwUuh5jiyX7ZVz/se49fH+debBITjEBclhrrUo4ieG3dMBHIydW5GdH0bXscuBtu1ooASUV3kCHtHDwlpNS1bbM3cX1aI5tZuBKkkNT5U7nu4AUspOjGW+rUoxdyGVEN2BL1Y1yKRaO6EnuuVUuhislBy8q4tDyU5hQsr6+a3x3tB2KDtAlr2fOIFouzHvt5i6x/Td+Cu2p9uD7iSeSAnuqyW2SoMN3f76mVaq200F715GUc45Kyu2xSfJBT5wjjwByCB6W/3YiOe6/tk1uVLJnsFe1ekcKOcpFweQJR4VGbbSBUNybKiLR505WQGYBstdPfEtw7cVjA1UBeLR7LHLVE63WN93AbRbgZqma6f4SJ3Gn2JBuIHQAr4CdlhvV/jsZuSOJRufzoKNfHUzumAdxbQuwf0g7HqB2APxPaips+NHXtHw176RmVJGntNMcv+TWmdO37f71ue1QYhG+ki9UV4mXwvpKywQGUsN/7qWjD5bLPTZw+LcSpy42M0v1+mRWeXdGCt/hVNZbFx1NBatemI86rVPsCPCyLIY0tSqBGyLI9vGyLYqsu0wpMxVuwi35Xu4VLXdPU2x/i9QSwMEFAAAAAgAm7xRXZtm4UXOAgAAlQYAABIAAABkZXRlY3Rpb24vdHJhaW4ucHmFVMFqGzEQvRv8D4N6yC64GxJ6cnGhmIYUklKStBdjxFirtUV3JSPJTral0I/oF/ZLOtLu2mvHaXWb0cybN29GYoxNbz6Ct6i00ktwwqq1h8JY8CsJC3SyVFpCLr0U3thsOBgOvjhcyvFwAHTWtV8Z3d4ro88jVLauYVaZXJY8V3YOsxw9QfloBYiHEOUA4Q51bqorY6Xz+3KFNRVcT+9uYIr29TWKb4Hc9P4rpegcHG6lCwSHg1gE0HpVoPAjqKS3SrhRjCMDQ2XwBnZsIMllgZvSQ3YenS6lrhhjgZeq1sZ6QFdroczOdrUbDiKpNfpVqRbQXnwmM+TFuwXxlDrPnLRbJaTLmpqt2aXcBud94wupw0GsRhIWUJEqSdpKu2c8iXUSYpGhXW5nF/MUVAGl1DtfCu/gAqgZ2cSyrjmWNmi9CTzDuzyNd3mIFxDOLT52iF1fk4OWkh3tifN2b6WjPoV417PTNAgRUF/Bg63DwEqDOU0R5JNyPow/Qr0NRLWhHYl+GnRcuGZjaHvRi1XWIHlbt0qGg4+ofMc5o233Ckv1XSbpPmZtlfZJwW6bpSqtxLyORGTeVPix6+dnBp8MWLl7OlpKCss6deSTkPSS7jbaq0p+sJaeFLrg7pGiXhihRAwq0SxzYTY6Z0BNBZEoIe1lhFOa5VKGKXLe7BTnCQtOosHSbCn9TYxI2NYZkUV0lp7CIB0Kk7xAAf78+r3/GQ4EPkY7FJfHJN695uQouH2ixL9LIMa89R4Hk0LtzZEI+3k994dTsIeOujDVuqQPKoP3QmwsinpMg2xQZ2fY+s7m4+xN8XME7CXAqwt6oMKafnZxwaPv/9kOAws3biSd7CE6iXkbcTb/F4ynj7KfTGYv8UTakaDhSb8o5SnNaDm6ieEWVYmLUh4swAlEQnHt50YD5FxjJTmHyQQY5+GT45y1Ke1Xm9mNTprvL/0LUEsDBBQAAAAIABwSUl1a3hP2IgcAAA4XAAAbAAAAc2NyaXB0cy92YWxpZGF0ZV9kYXRhc2V0LnB5tVhbb9s2FH7XryD0JGG26uxpMOoCXuJsBdIsaNKugGcIjEQn3HQrSSVxg/z3nUOKkijJQfawPMS2zpXn+lG+718rwWjOizvy++nnC5JSRSVT5IFmHL7zsiCPXN0TWhD2VGU84YrkNNuXImfpXJSPJOM5V5HnfZH0ji1JdVD3ICQTwSsl3zV6WNwojqoDeY/f45SLD2Q7n+f0ac4LzTcXwEkW0WJxsvN83/c8nlelUISKu4oKyezvv2VZ2O+ltN/kQXp7UeYkKbOMJei8JA3ttKwLxYShV1TdZ/zW0q7gpyGoQ4WRaJ6vi4Nnnt/S5B9WpJFk4oEnTEaVYJUo4Zvs8RcQFDjGDxbfJyKLM3rLshnRjscJLeKMF8zzNt+uNqc3m7P4Yv3r5uKarMizf6kl/Rnxz8pr/Divf/w44JffGBX4+fnqk//inX+82Bi5+NP6CkU9An8o1QY4kQ/+slFkqFrZkN5YMBx3YGXIYCwbOlgfktGhhmrOHYu60EyRelLIYQ/lvXhnm/P1l4sbcPpb/PHy6/ri41n8+Y8/48/rmw2cYp+VVAWljO6YYsVD4J+tb9bXm2l+jIYuET8MPc9L2Z60RbbnGQu0U3vM8VKndgYF+xQ3JRZjiS2NRbD8mmNeSOYfSMoTtZVKzLAcdkutHDoByoulJsWgxs0LHiPQDkQFzRn464daTjBZZ2o50NmmUQcTjwDR64l3NPbEpZIt1fwMwh4H9HJKb7WGc5rJvrDEwrw9KNYpkArCHsJHjETC9wPFhIEKsujrLx/jBFsJdPSfF2XB8kodYmCQA1oT9zGhTcmrJJ0xJEd9hu810MFeRaVk6cRxdWpkvAdnkQyV6wRKk81R0HYzHhwWbFmeyglhJkQpkLDdjZ7GCvogAZcHTr3o/xDiolSDMC9bHaZAttbALqJVBXMn2PvnUBZGFA+0JM9axUtTWEZW1aJoVHj6sRKHTrce48ZwCUoDH+cKK5IyhRG28mu1n/+CT7TpFVRSldGE+SGhksiyFgnrdPV9bUsOC/lG1MzhgkVBcO7FRZ3fMqh4QR/1ICQcNgo8ZJjewBiYkZPQNeIYamtvR35akZMRo1a7ai1AXcMOCsIRX5ME5Blbw7+kLBQvBifpu+KW+xF3nOj3//RGSMFTdzUE+G/srT5ZM2emNkxg1OmhY8raD2eDETWt1Z6m14VHjtLndhoLKjRNgzfYcPptt9U/32DM9qCxY0669am45UpodAJUfzc2zZ4SVikSfKVZzTZY0jPypeBQ7OaXrmpgmk6Qte4MqFe8hYLKoKOG3RuS9+RksZi20bczbvcLLOXnXuO8QMuDv/2Gd84Lc+atdnoTymnZJmYb/YHAbxii4+6e0gI7CkcBwQU2dnZiOqWsKAF4UgUjYnWstby+4dFW2PUEB8l65+iHDPV/6r12Ei0c7YOVgrqD1v++9AeyADycvubW+9UIdGhVoWPRbSWwJwFGsjSYpk/Itg0FsggqgmmqK9n21NhgS7ISTtJcrNXgQRMii+j/L8QFl4GvjWFw53vNBQzQhFbgH9MVJ2dmSEoDuGGlzXSO3LsKuhLhxaLpWizaxVSysHN7hU851Es3SwJ/xJ/XUpFbvdW2C9xjUPtaXtZ5TmETvAr6bPgQaygR2J/hEEUBHdu191ifHR4/v/Qe5lzfS2JLdGCKKhWC9THmMgRrZ0xpzvtfkZoTqo5pGMEJbLenPIP8jk5As2b1TWEzJE4iNwPAEI9gWBBZY7Jc5D6cdZAl925hMwPjxWoZF3s39Zrkb5s07bZWqBtcLW9Tju2QNVh/5071VqGb4m4UN/rDaSmTQDSukWnLxLKB9Q7UHbH/Zk3DqXpE3yjn7ZGeHQGdY3tHshkYcwyuFtM7YkJueI85spVet2hL/Ph+cOVf3pitcVn1elnDkwmofEzIKu9J9WNzTMwOAUfQDetYtNewu6iusJ1e3XEj4Q4GDqR7C0tLOnnC/VbnuifGkN0OAhwCwzaNHnDUw80s9BxXJvHH0QC9w8EQOHZxMXhHQ9Pt40nyhOTEKp+mh3bj4UCc1k7mZPBqajlOxlTbNyih4WlgQk55Eeh1flnaa5aG74j17Ou8aC3u4AZYqCtNCVJmXhsC+Fw1Lybndv6a15On60u79mWDLo1WvB/EtFEXdAt1hu/02ApRyXH2bnuMX0daDRq/zAAH7ilU3uo1GNN19z3LqpX/iT7xvM47LNKWqQUlvCSQi/IRSBVEyAKboLG2NO9Ew+aFmzkIeC/tHVJE5iqJzwKnTvobzCI25IpshGZaUTS9wyrBITz4tjVK67yStnRm0DWAiNXq57CtLBz6wzrpKsho8v8qdKjWNx//uCTna1i+ZxBgbLuVPEi4sadwuejdGjTwuj5IxfLNE1fBSd8vV9vV+vp6WpvngXtxjKsijslqRfw4xvqMY984aIrV+xdQSwMEFAAAAAgAElFSXQAAAAACAAAAAAAAABMAAABiYWNrZW5kL19faW5pdF9fLnB5AwBQSwMEFAAAAAgAElFSXQAAAAACAAAAAAAAABwAAABiYWNrZW5kL3NlcnZpY2VzL19faW5pdF9fLnB5AwBQSwMEFAAAAAgAElFSXQAAAAACAAAAAAAAABUAAABkZXRlY3Rpb24vX19pbml0X18ucHkDAFBLAwQUAAAACAASUVJdAAAAAAIAAAAAAAAAEwAAAHNjcmlwdHMvX19pbml0X18ucHkDAFBLAQIUABQAAAAIACMSUl1UUnmaZBkAAAlgAAAhAAAAAAAAAAAAAAC2gQAAAABiYWNrZW5kL3NlcnZpY2VzL21vZGVsX3NlcnZpY2UucHlQSwECFAAUAAAACAADElJd+PGWR/UFAACWDwAAIQAAAAAAAAAAAAAAtoGjGQAAYmFja2VuZC9zZXJ2aWNlcy9wcmVwcm9jZXNzaW5nLnB5UEsBAhQAFAAAAAgAh55RXfm1pkl0CQAAzyEAABUAAAAAAAAAAAAAALaB1x8AAGRldGVjdGlvbi9mZWF0dXJlcy5weVBLAQIUABQAAAAIAJu8UV2bZuFFzgIAAJUGAAASAAAAAAAAAAAAAAC2gX4pAABkZXRlY3Rpb24vdHJhaW4ucHlQSwECFAAUAAAACAAcElJdWt4T9iIHAAAOFwAAGwAAAAAAAAAAAAAAtoF8LAAAc2NyaXB0cy92YWxpZGF0ZV9kYXRhc2V0LnB5UEsBAhQAFAAAAAgAElFSXQAAAAACAAAAAAAAABMAAAAAAAAAAAAAAIAB1zMAAGJhY2tlbmQvX19pbml0X18ucHlQSwECFAAUAAAACAASUVJdAAAAAAIAAAAAAAAAHAAAAAAAAAAAAAAAgAEKNAAAYmFja2VuZC9zZXJ2aWNlcy9fX2luaXRfXy5weVBLAQIUABQAAAAIABJRUl0AAAAAAgAAAAAAAAAVAAAAAAAAAAAAAACAAUY0AABkZXRlY3Rpb24vX19pbml0X18ucHlQSwECFAAUAAAACAASUVJdAAAAAAIAAAAAAAAAEwAAAAAAAAAAAAAAgAF7NAAAc2NyaXB0cy9fX2luaXRfXy5weVBLBQYAAAAACQAJAHkCAACuNAAAAAA="
REQUIRED_DATA_FILES = {
    "DoS_dataset.csv", "Fuzzy_dataset.csv", "gear_dataset.csv",
    "RPM_dataset.csv", "normal_run_data.txt",
}

def is_project_root(path):
    return (path / "backend" / "services" / "model_service.py").is_file() and         (path / "detection" / "train.py").is_file()

cwd = Path.cwd().resolve()
PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT", str(cwd / "ZeroSight"))).expanduser().resolve()
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE_B64))) as embedded_project:
    embedded_project.extractall(PROJECT_ROOT)
if not is_project_root(PROJECT_ROOT):
    raise RuntimeError(f"Embedded project bundle is incomplete: {PROJECT_ROOT}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

def has_all_dataset_files(path):
    return path.is_dir() and REQUIRED_DATA_FILES.issubset({item.name for item in path.iterdir()})

configured_dataset = os.getenv("DATASET_DIR")
if configured_dataset:
    DATASET_DIR = Path(configured_dataset).expanduser().resolve()
elif has_all_dataset_files(PROJECT_ROOT / "data" / "raw"):
    DATASET_DIR = PROJECT_ROOT / "data" / "raw"
else:
    search_roots = [Path("/kaggle/input"), Path("/content"), cwd]
    DATASET_DIR = None
    searched = set()
    for search_root in search_roots:
        if not search_root.exists():
            continue
        for candidate in search_root.rglob("DoS_dataset.csv"):
            parent = candidate.parent.resolve()
            if parent in searched:
                continue
            searched.add(parent)
            if has_all_dataset_files(parent):
                DATASET_DIR = parent
                break
        if DATASET_DIR:
            break
if not DATASET_DIR or not has_all_dataset_files(DATASET_DIR):
    raise FileNotFoundError(
        "Could not locate all five HCRL dataset files. Upload/mount them together "
        "in one folder, then set DATASET_DIR to that folder before running this cell. "
        f"Required files: {sorted(REQUIRED_DATA_FILES)}"
    )

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
default_model_dir = PROJECT_ROOT / "models" / "candidates" / run_id
MODEL_DIR = Path(os.getenv("MODEL_DIR", str(default_model_dir)))
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"Embedded project: {PROJECT_ROOT}")
print(f"Dataset:    {DATASET_DIR}")
print(f"Candidate:  {MODEL_DIR}")


## 2. Stream-validate the dataset

The validator reads the five expected HCRL files line by line. Its report
includes byte sizes, physical/nonempty/valid/invalid row counts, normalized
label counts, CAN IDs, and representative parse errors. The default
per-capture malformed-row limit is 0.1%; override it only with an approved
quality threshold. Training runs this same gate before reading data.


In [ ]:
from scripts.validate_dataset import validate_dataset, DEFAULT_MAX_INVALID_ROW_RATE

validation = validate_dataset(
    DATASET_DIR,
    max_invalid_rate=DEFAULT_MAX_INVALID_ROW_RATE,
)
print(json.dumps(validation, indent=2))
if not validation["valid"]:
    raise RuntimeError("Dataset validation failed. Do not train this candidate.")
validation_path = MODEL_DIR / "dataset_validation.json"
validation_path.write_text(json.dumps(validation, indent=2), encoding="utf-8")
print(f"Validation report: {validation_path}")


### Dataset quality and label distribution

The first panel compares valid and malformed rows by capture. The second
shows the row-level class distribution reported by the validator.


In [ ]:
import matplotlib.pyplot as plt

capture_names = list(validation["files"])
valid_counts = [validation["files"][name]["valid_rows"] for name in capture_names]
invalid_counts = [validation["files"][name]["invalid_rows"] for name in capture_names]
label_counts = {}
for file_result in validation["files"].values():
    for label, count in file_result["label_counts"].items():
        label_counts[label] = label_counts.get(label, 0) + count

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
positions = range(len(capture_names))
axes[0].bar(positions, valid_counts, label="Valid rows", color="#2a9d8f")
axes[0].bar(positions, invalid_counts, bottom=valid_counts, label="Malformed rows", color="#e76f51")
axes[0].set_xticks(list(positions), capture_names, rotation=35, ha="right")
axes[0].set_ylabel("Rows")
axes[0].set_title("Rows by capture")
axes[0].legend()
axes[1].bar(list(label_counts), list(label_counts.values()), color="#457b9d")
axes[1].set_ylabel("Rows")
axes[1].set_title("Normalized row labels")
axes[1].tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()


## 3. Train an isolated candidate with the CLI

This cell invokes `detection/train.py` in a subprocess. The default candidate
directory has a unique UTC timestamp; set `MODEL_DIR` before opening the
notebook to choose a different isolated location. Run this after the validation
report is clean. The CLI refuses data that fails the quality gate:

```sh
python detection/train.py ./models/candidate ./data/raw
```

The CLI saves the model, metrics, feature schema, training-only reference IDs,
split provenance, and SHA-256 artifact hash for API startup verification.


In [ ]:
if (MODEL_DIR / "baseline_rf.pkl").exists():
    raise RuntimeError(f"Candidate already has a model; choose a fresh MODEL_DIR: {MODEL_DIR}")

training_command = [
    sys.executable,
    "-m",
    "detection.train",
    str(MODEL_DIR),
    str(DATASET_DIR),
]
training_started = time.perf_counter()
training_result = subprocess.run(
    training_command, cwd=PROJECT_ROOT, text=True, capture_output=True
)
training_runtime_seconds = time.perf_counter() - training_started
if training_result.stdout:
    print(training_result.stdout)
if training_result.stderr:
    print(training_result.stderr)
if training_result.returncode:
    raise RuntimeError(f"Training CLI failed with exit code {training_result.returncode}")

expected_artifacts = ["baseline_rf.pkl", "metrics.json", "metadata.json"]
missing_artifacts = [name for name in expected_artifacts if not (MODEL_DIR / name).is_file()]
if missing_artifacts:
    raise RuntimeError(f"Training did not create expected artifacts: {missing_artifacts}")
print(f"Training runtime: {training_runtime_seconds:.2f} seconds")


## 4. Load the candidate and verify its schema

Point `MODEL_DIR` at the candidate directory. A new `ModelService` instance
performs the same artifact hash and feature-schema checks as API startup.


In [ ]:
from backend.services.model_service import ModelService
from detection.features import FEATURE_VERSION, WindowFeatureExtractor

service = ModelService(model_dir=str(MODEL_DIR), dataset_dir=str(DATASET_DIR))
await service.initialize()

metadata = service.model_metadata
extractor = WindowFeatureExtractor(
    window_ms=service.window_ms,
    reference_can_ids=set(metadata["reference_can_ids"]),
)
metrics = service.get_metrics()
assert metadata["feature_version"] == FEATURE_VERSION
assert metadata["feature_names"] == extractor.feature_names()
assert metadata["split_strategy"] == "capture_aware_chronological_80_20"
assert metadata["training_samples"] == metrics["training_samples"]
assert metadata["test_samples"] == metrics["test_samples"]

print(json.dumps(metadata, indent=2))
print(f"Feature schema: {FEATURE_VERSION} ({len(metadata['feature_names'])} columns)")


## 5. Held-out evaluation metrics

These metrics are computed from the chronological test tail of each capture.
They are actual model results; API load and per-window ground-truth metrics
must be measured separately against the deployed service.


In [ ]:
if not metrics:
    raise RuntimeError("The candidate has no stored evaluation metrics.")
print(json.dumps(metrics, indent=2))

# Plot observed held-out model results from the stored evaluation.
import numpy as np
import matplotlib.pyplot as plt

classes = metrics["classes"]
cm = np.asarray(metrics["confusion_matrix"])
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
image = axes[0].imshow(cm, cmap="Blues")
axes[0].set_xticks(range(len(classes)), classes, rotation=35, ha="right")
axes[0].set_yticks(range(len(classes)), classes)
axes[0].set_xlabel("Predicted label")
axes[0].set_ylabel("True label")
axes[0].set_title("Held-out confusion matrix")
for row in range(cm.shape[0]):
    for col in range(cm.shape[1]):
        axes[0].text(col, row, int(cm[row, col]), ha="center", va="center", color="black")
fig.colorbar(image, ax=axes[0], fraction=0.046, pad=0.04)

per_class = metrics["per_class"]
metric_names = ["precision", "recall", "f1"]
x = np.arange(len(classes))
width = 0.24
for index, metric_name in enumerate(metric_names):
    values = [per_class.get(label, {}).get(metric_name, 0.0) for label in classes]
    axes[1].bar(x + (index - 1) * width, values, width, label=metric_name.title())
axes[1].set_xticks(x, classes, rotation=35, ha="right")
axes[1].set_ylim(0, 1)
axes[1].set_ylabel("Score")
axes[1].set_title("Per-class held-out metrics")
axes[1].legend()
fig.tight_layout()
plt.show()

if hasattr(service.model, "feature_importances_"):
    importances = service.model.feature_importances_
    top_indices = np.argsort(importances)[-15:]
    feature_names = [metadata["feature_names"][index] for index in top_indices]
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(feature_names, importances[top_indices], color="#6a4c93")
    ax.set_xlabel("Random Forest importance")
    ax.set_title("Top 15 window features")
    fig.tight_layout()
    plt.show()


## 6. Verify reload and inference in a fresh process

This starts another Python interpreter and loads the persisted artifact from
disk. A successful training run alone does not establish that API startup can
restore the feature schema or perform inference.


In [ ]:
fresh_process_code = r'''import asyncio, json, sys
from backend.services.model_service import ModelService

async def main():
    service = ModelService(model_dir=sys.argv[1], dataset_dir=sys.argv[2])
    await service.initialize()
    frames = [{
        "timestamp": 1760000000.0 + index * 0.001,
        "arbitration_id": "0316", "dlc": 8,
        "data_bytes": [5, 33, 104, 9, 33, 33, 0, 111],
    } for index in range(8)]
    prediction, confidence, features = await service.predict_window(frames)
    print(json.dumps({
        "prediction": prediction,
        "confidence": confidence,
        "feature_count": len(features),
        "model_version": service.model_version,
        "artifact_hash": service.model_metadata["artifact_hash"],
        "frame_count": len(frames),
    }))

asyncio.run(main())
'''
fresh_result = subprocess.run(
    [sys.executable, "-c", fresh_process_code, str(MODEL_DIR), str(DATASET_DIR)],
    cwd=PROJECT_ROOT,
    check=True,
    text=True,
    capture_output=True,
)
fresh_process_inference = json.loads(fresh_result.stdout.strip().splitlines()[-1])
print(json.dumps(fresh_process_inference, indent=2))


## 7. Predict from a complete CAN window

`POST /detect/window` passes all frames from one bounded window to a single
model inference call. This cell exercises the same `predict_window` method on
eight frames within 100 ms. The timer is model-call duration, not API round-trip
latency.


In [ ]:
sample_frames = [{
    "timestamp": 1760000000.0 + index * 0.001,
    "arbitration_id": "0316",
    "dlc": 8,
    "data_bytes": [5, 33, 104, 9, 33, 33, 0, 111],
} for index in range(8)]

started = time.perf_counter()
prediction, confidence, features = await service.predict_window(sample_frames)
inference_latency_ms = (time.perf_counter() - started) * 1000
print(json.dumps({
    "prediction": prediction,
    "confidence": confidence,
    "inference_latency_ms": round(inference_latency_ms, 4),
    "frame_count": len(sample_frames),
    "feature_count": len(features),
    "model_version": service.model_version,
    "feature_version": metadata["feature_version"],
}, indent=2))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(["Prediction confidence"], [confidence], color="#2a9d8f")
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("Confidence")
axes[0].set_title(f"Window prediction: {prediction}")
axes[1].bar(["Inference"], [inference_latency_ms], color="#e9c46a")
axes[1].set_ylabel("Milliseconds")
axes[1].set_title("Single-window model-call latency")
fig.tight_layout()
plt.show()

verification_report = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "dataset_validation": validation,
    "training_runtime_seconds": round(training_runtime_seconds, 4),
    "model_metadata": metadata,
    "metrics": metrics,
    "fresh_process_inference": fresh_process_inference,
    "notebook_inference": {
        "prediction": prediction,
        "confidence": confidence,
        "inference_latency_ms": round(inference_latency_ms, 4),
        "frame_count": len(sample_frames),
        "feature_count": len(features),
    },
}
report_path = MODEL_DIR / "notebook_verification_report.json"
report_path.write_text(json.dumps(verification_report, indent=2), encoding="utf-8")
print(f"Verification report saved: {report_path}")


## Scope

This notebook covers data quality, CLI training, model reload, evaluation,
inference, and an ML verification report. API authentication, privacy and
dual-approval flows, and deployment load testing remain separate service-level
checks. Use `tests/load/loadtest.py` against the target API for those checks.
